# Train **llmscratch** on Google Colab

Trains the from-scratch LLM on a Colab GPU using the **PyTorch/CUDA track** (`train_torch`). On CUDA it auto-enables bf16 + `torch.compile` + TF32 + fused AdamW.

**First:** `Runtime -> Change runtime type -> GPU` (T4 is free; L4/A100 on Colab Pro).

Colab runtimes are ephemeral (idle / ~12 h limits), so we save checkpoints to Google Drive and use `--resume` — re-running the training cell after a disconnect continues seamlessly.

In [ ]:
!nvidia-smi

## 1. Get the code + install

In [ ]:
!git clone https://github.com/wishbone305/llm_scratch.git
%cd llm_scratch
!pip install -q -e .   # torch is preinstalled on Colab; mlx auto-skips (platform marker)

## 2. Persist data + checkpoints on Google Drive
So they survive disconnects. `data/` is symlinked to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/llmscratch/data', exist_ok=True)
os.makedirs('/content/drive/MyDrive/llmscratch/out', exist_ok=True)
!rm -rf data && ln -s /content/drive/MyDrive/llmscratch/data data
OUT = '/content/drive/MyDrive/llmscratch/out'
!ls -la data/

## 3. Get the data — pick ONE

**Option A (fastest):** upload your `train.bin` + `val.bin` to `MyDrive/llmscratch/data/` once (they persist across sessions). Then skip Option B.

**Option B:** stream FineWeb-Edu and tokenize on Colab (~1B tokens, several minutes).

In [ ]:
# Option B only — skip if you uploaded bins to Drive (they'd already be in data/).
!pip install -q datasets
from datasets import load_dataset
from llmscratch.data import write_documents
import itertools
it = iter(load_dataset('HuggingFaceFW/fineweb-edu', 'sample-10BT',
                       split='train', streaming=True))
# islice advances the shared iterator, so val is held out from train (no overlap)
n_tr = write_documents((ex['text'] for ex in itertools.islice(it, 500_000)), 'data/train.bin')
n_va = write_documents((ex['text'] for ex in itertools.islice(it, 4_000)),   'data/val.bin')
print(f'train {n_tr:,} tokens | val {n_va:,} tokens')

## 4. Train

Pick a config for your GPU:
- **A100 / L4 (Colab Pro):** `gpt_200m` — the full ~205M target (~10-12 h on A100).
- **Free T4 (16 GB):** `fineweb_125m` — fits via gradient checkpointing.

`--resume` continues from Drive after any disconnect (just re-run this cell).

In [ ]:
CONFIG = 'gpt_200m'   # free T4 -> use 'fineweb_125m'
!python -m llmscratch.train_torch --config {CONFIG} --data-dir data --out-dir {OUT} --resume

## 5. Watch training (TensorBoard)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 6. Sample from a checkpoint
Base model -> prompt with the *start* of something (top-p + repetition-penalty are on by default).

In [ ]:
!python -m llmscratch.sample_torch --ckpt {OUT}/{CONFIG}.pt \
    --prompt 'The Sun is a star that' --max-new-tokens 150

---
### Reconnect workflow
If Colab disconnects: reconnect, re-run cells **1, 2, 4** (skip Option B — your bins are on Drive), and `--resume` picks up from the last checkpoint. Keep the tab active to avoid idle timeouts. For long unattended runs, Colab Pro or a dedicated host (Lambda / RunPod) is steadier.